In [4]:
import cvxpy as cp
import numpy as np

# Data
# 5 Warehouses, 10 Stores
num_warehouses = 5
num_stores = 10

fixed_costs = [180, 550, 220, 240, 120]

# Transportation Costs Matrix (Rows=Warehouses, Cols=Stores)
trans_costs = np.array([
    [65, 85, 55, 40, 50, 85, 60, 50, 75, 30],  # W1
    [35, 55, 20, 50, 75, 55, 40, 25, 45, 55],  # W2
    [40, 50, 65, 25, 55, 80, 25, 35, 20, 25],  # W3
    [45, 80, 45, 65, 25, 50, 70, 65, 45, 55],  # W4
    [60, 50, 40, 45, 25, 30, 35, 60, 25, 45]   # W5
])

# Variables
y = cp.Variable(num_warehouses, boolean=True)
x = cp.Variable((num_warehouses, num_stores), boolean=True)

constraints = []

# Constraint 1: Each store serves exactly 1 warehouse
for j in range(num_stores):
    constraints.append(cp.sum(x[:, j]) == 1)

# Constraint 2: Capacity (Max 4 stores) & Linking
# If y[i] is 0, sum(x[i,:]) must be 0. If y[i] is 1, sum can be up to 4.
for i in range(num_warehouses):
    constraints.append(cp.sum(x[i, :]) <= 4 * y[i])

# Objective
total_fixed = cp.sum(cp.multiply(fixed_costs, y))
total_trans = cp.sum(cp.multiply(trans_costs, x))

prob = cp.Problem(cp.Minimize(total_fixed + total_trans), constraints)

print("Solving...")
prob.solve(solver=cp.MOSEK)

print("Status:", prob.status)
print("Total Annual Cost:", prob.value)

print("\nSolution:")
for i in range(num_warehouses):
    if y[i].value > 0.5: # If open
        print(f"Warehouse {i+1} is OPEN.")
        stores_served = []
        for j in range(num_stores):
            if x[i, j].value > 0.5:
                stores_served.append(f"S{j+1}")
        print(f"  Serves: {stores_served}")

Solving...
Status: optimal
Total Annual Cost: 855.0

Solution:
Warehouse 1 is OPEN.
  Serves: ['S8', 'S10']
Warehouse 3 is OPEN.
  Serves: ['S1', 'S4', 'S7', 'S9']
Warehouse 5 is OPEN.
  Serves: ['S2', 'S3', 'S5', 'S6']
